# E17: Vanilla LSTM Baseline (All 24 Features)

Single LSTM on all 24 features (no multi-agent decomposition). Same hyperparameters as E2 (32 hidden, 1 layer). Same train/val/test split. Reuses existing mmap data on Drive — no preprocessing needed.

**Purpose**: Address Maoying #5 (fair temporal baseline) and the most damaging defence question ("is your multi-agent design actually better than a single LSTM?").

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, gc, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix
)
import matplotlib.pyplot as plt

PROJECT_PATH = '/content/drive/MyDrive/Sepsis'
MODEL_PATH = f'{PROJECT_PATH}/models'
MMAP_DIR = f'{MODEL_PATH}/E2/mmap_cache'  # reuse E2's mmap (same seq_length=24)
LOCAL_CACHE = '/content/cache/E17'
SAVE_DIR = f'{MODEL_PATH}/E17'
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Copy E2 mmap to local SSD for speed
if not os.path.exists(LOCAL_CACHE):
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    print('Copying mmap files to local SSD...')
    for split in ['train', 'val', 'test']:
        for suffix in ['_seqs.npy', '_labels.npy', '_masks.npy']:
            src = f'{MMAP_DIR}/{split}{suffix}'
            dst = f'{LOCAL_CACHE}/{split}{suffix}'
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                size_gb = os.path.getsize(dst) / 1e9
                print(f'  {split}{suffix}: {size_gb:.1f} GB')
    print('Done.')
else:
    print(f'Local cache exists at {LOCAL_CACHE}')

In [ ]:
# Load mmap arrays
train_seqs = np.load(f'{LOCAL_CACHE}/train_seqs.npy', mmap_mode='r')
train_labels = np.load(f'{LOCAL_CACHE}/train_labels.npy', mmap_mode='r')
train_masks = np.load(f'{LOCAL_CACHE}/train_masks.npy', mmap_mode='r')
val_seqs = np.load(f'{LOCAL_CACHE}/val_seqs.npy', mmap_mode='r')
val_labels = np.load(f'{LOCAL_CACHE}/val_labels.npy', mmap_mode='r')
val_masks = np.load(f'{LOCAL_CACHE}/val_masks.npy', mmap_mode='r')
test_seqs = np.load(f'{LOCAL_CACHE}/test_seqs.npy', mmap_mode='r')
test_labels = np.load(f'{LOCAL_CACHE}/test_labels.npy', mmap_mode='r')
test_masks = np.load(f'{LOCAL_CACHE}/test_masks.npy', mmap_mode='r')

print(f'Train: {train_seqs.shape}')
print(f'Val:   {val_seqs.shape}')
print(f'Test:  {test_seqs.shape}')
print(f'Features per timestep: {train_seqs.shape[2]}')  # should be 24

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, seqs, labels, masks):
        self.seqs = seqs
        self.labels = labels
        self.masks = masks
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        # all features, with NaN -> 0 (same as multi-agent labs handling)
        x = np.nan_to_num(self.seqs[idx], nan=0.0)
        return torch.from_numpy(x).float(), float(self.labels[idx])

BATCH_SIZE = 1024
train_loader = DataLoader(SequenceDataset(train_seqs, train_labels, train_masks),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader = DataLoader(SequenceDataset(val_seqs, val_labels, val_masks),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
                        pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(SequenceDataset(test_seqs, test_labels, test_masks),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
                         pin_memory=True, persistent_workers=True, prefetch_factor=4)

In [ ]:
class VanillaLSTM(nn.Module):
    """Single Bi-LSTM on all 24 features. Same hidden dim as E2's per-agent LSTM.
    
    Compared to MultiAgentSepsisPredictor (E2):
    - Same input data (24 features per timestep)
    - Same Bi-LSTM hidden dim (32) and layer count (1)
    - Same dropout (0.3)
    - But no agent decomposition, no learned imputation, no Transformer
    - Just one LSTM + attention + classifier
    """
    def __init__(self, input_dim=24, hidden_dim=32, num_layers=1, dropout=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        x = self.norm(x)
        out, _ = self.lstm(x)
        attn = torch.softmax(self.attention(out), dim=1)
        ctx = (attn * out).sum(dim=1)
        return self.classifier(ctx).squeeze(-1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = targets * p + (1 - targets) * (1 - p)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()

model = VanillaLSTM(input_dim=24, hidden_dim=32, num_layers=1, dropout=0.3).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Vanilla LSTM parameters: {n_params:,}')
print(f'(For reference: Multi-Agent E2 has ~56,200 parameters)')

In [ ]:
criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
scaler = torch.cuda.amp.GradScaler()

def evaluate(loader):
    model.eval()
    losses, preds, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = criterion(logits, y)
            losses.append(loss.item())
            preds.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(y.cpu().numpy())
    preds = np.concatenate(preds); labels = np.concatenate(labels)
    auroc = roc_auc_score(labels, preds)
    auprc = average_precision_score(labels, preds)
    return np.mean(losses), auroc, auprc, preds, labels

In [ ]:
EPOCHS = 30
PATIENCE = 5
best_val_auroc = 0
patience_counter = 0
history = {'train_loss': [], 'val_auroc': [], 'val_auprc': []}

start = time.time()
for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    train_losses = []
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())
    
    val_loss, val_auroc, val_auprc, _, _ = evaluate(val_loader)
    scheduler.step(val_auroc)
    history['train_loss'].append(np.mean(train_losses))
    history['val_auroc'].append(val_auroc)
    history['val_auprc'].append(val_auprc)
    
    epoch_time = time.time() - epoch_start
    flag = ''
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict(), 'val_auroc': val_auroc},
                   f'{SAVE_DIR}/best_model.pt')
        flag = ' *BEST*'
    else:
        patience_counter += 1
        flag = f' (patience {patience_counter}/{PATIENCE})'
    print(f'  Epoch {epoch+1:2d} | Train Loss: {np.mean(train_losses):.4f} | '
          f'Val AUROC: {val_auroc:.4f} | Val AUPRC: {val_auprc:.4f} | '
          f'{epoch_time:.0f}s{flag}')
    if patience_counter >= PATIENCE:
        print(f'  Early stopping at epoch {epoch+1}')
        break

elapsed = (time.time() - start) / 60
print(f'\nTraining complete: {elapsed:.1f} min')

In [ ]:
# Load best, evaluate on test
ckpt = torch.load(f'{SAVE_DIR}/best_model.pt', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
test_loss, test_auroc, test_auprc, test_preds, test_labels_arr = evaluate(test_loader)

# Optimal threshold
prec, rec, thresholds = precision_recall_curve(test_labels_arr, test_preds)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
opt_idx = np.argmax(f1_scores)
opt_threshold = float(thresholds[opt_idx]) if opt_idx < len(thresholds) else 0.5
best_f1 = float(f1_scores[opt_idx])
test_pred_binary = (test_preds >= opt_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(test_labels_arr, test_pred_binary).ravel()
sens = tp / (tp + fn) if (tp + fn) > 0 else 0
spec = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print('=' * 60)
print('E17 VANILLA LSTM TEST RESULTS')
print('=' * 60)
print(f'  AUROC:       {test_auroc:.4f}')
print(f'  AUPRC:       {test_auprc:.4f}')
print(f'  F1:          {best_f1:.4f} (threshold={opt_threshold:.3f})')
print(f'  Sensitivity: {sens:.4f}')
print(f'  Specificity: {spec:.4f}')
print(f'  PPV:         {ppv:.4f}')
print(f'  NPV:         {npv:.4f}')
print(f'  Time:        {elapsed:.1f} min')
print()
print('=' * 60)
print('COMPARISON TO MULTI-AGENT E2')
print('=' * 60)
E2 = {'auroc': 0.7689, 'auprc': 0.7008, 'f1': 0.7136}
print(f'  AUROC: E17={test_auroc:.4f}  E2={E2["auroc"]:.4f}  Δ={test_auroc-E2["auroc"]:+.4f}')
print(f'  AUPRC: E17={test_auprc:.4f}  E2={E2["auprc"]:.4f}  Δ={test_auprc-E2["auprc"]:+.4f}')
print(f'  F1:    E17={best_f1:.4f}  E2={E2["f1"]:.4f}  Δ={best_f1-E2["f1"]:+.4f}')

In [ ]:
# Save results.json + test predictions
results = {
    'version': 'E17',
    'description': 'Vanilla Bi-LSTM baseline on all 24 features',
    'test_auroc': float(test_auroc),
    'test_auprc': float(test_auprc),
    'test_loss': float(test_loss),
    'f1': float(best_f1),
    'optimal_threshold': opt_threshold,
    'sensitivity': float(sens),
    'specificity': float(spec),
    'ppv': float(ppv),
    'npv': float(npv),
    'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]],
    'n_params': int(n_params),
    'training_time_min': round(elapsed, 1),
    'history': history,
    'config': {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3,
               'learning_rate': 1e-4, 'batch_size': 1024, 'epochs': EPOCHS,
               'patience': PATIENCE, 'focal_alpha': 0.25, 'focal_gamma': 2.0,
               'weight_decay': 1e-4, 'sequence_length': 24}
}
with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)
np.savez(f'{SAVE_DIR}/test_predictions.npz', preds=test_preds, labels=test_labels_arr)
print(f'Saved to {SAVE_DIR}/')

# Auto-cleanup local cache
if os.path.exists(LOCAL_CACHE):
    cache_size_gb = sum(os.path.getsize(os.path.join(LOCAL_CACHE, f))
                        for f in os.listdir(LOCAL_CACHE)) / 1e9
    shutil.rmtree(LOCAL_CACHE, ignore_errors=True)
    print(f'Cleaned up local cache ({cache_size_gb:.1f} GB freed)')